# 1. Load and convert labeled data into sequential datasets

path/to/labeled_data/ 

    ├── class_0/
    │   ├── sample1.pkl
    │   ├── sample2.csv
    │   └── ...
    ├── class_1/
    │   ├── sample1.csv
    │   ├── sample2.pkl
    │   └── ...
    └── ...

dict_label={'class_0':0, 'class_1':1, ... }

In [1]:
import numpy as np
import os
import pandas as pd
from torch.utils.data import Dataset
import torch
from torchinfo import summary 
import random
from torch.utils.data import random_split, DataLoader, Subset, ConcatDataset


class LoadSeqDataset(Dataset):
    def __init__(self, file_path: str, label: int, selected_features:list, seq_num=28, gap=5, desire_class = None):
        """
        Initialize the dataset from a labeled data file by converting it to sequential format.
        
        Args:
            file_path (str): Path to the labeled data file.
            label (int): Label associated with the sequences from this file.
            seq_num (int): Length of each sequence.
            gap (int): Step size between sequences.
        """
        self.seq_num = seq_num
        self.gap = gap
        self.selected_features = selected_features
        # Define dtype for selected features and label
        dtypes = {col: 'float32' for col in selected_features}
        dtypes['label'] = 'int8'  # Set label as an integer type
        dtypes['time'] = 'float32'  # Set label as an integer type
        # Load data and process into sequences

        self.data = pd.read_csv(file_path, usecols=selected_features + ['label', 'time'], dtype = dtypes)
        self.data.drop(columns='index', inplace=True, errors='ignore')
        self.data.label = self.data.label * label.item()
        self.df = self.data
        #self.sequences = self._make_sequences_contact_only()
        self.sequences = self._make_sequences()
        if desire_class is not None:
            self.sequences = self._get_specific_class(desire_class)
    def balanceData(self, split_rate_1 = 1, split_rate_0=0.15):
        """Balances the dataset by downsampling sequences where label = 0 to 10%."""
        # Separate sequences based on the label
        label_0_sequences = [seq for seq in self.sequences if seq[1] == 0]
        other_label_sequences = [seq for seq in self.sequences if seq[1] != 0]

        # Downsample label 0 sequences to 10%
        num_to_keep = int(len(label_0_sequences) * split_rate_0)
        downsampled_label_0 = random.sample(label_0_sequences, num_to_keep)

        num_to_keep = int(len(other_label_sequences) * split_rate_1)
        other_label_sequences = random.sample(other_label_sequences, num_to_keep)

        # Combine and shuffle
        balanced_sequences = downsampled_label_0 + other_label_sequences
        random.shuffle(balanced_sequences)

        # Update sequences
        self.sequences = balanced_sequences

    def _make_sequences_contact_only(self):
        """Generate sequences based on the contact points detected in the data."""
        start_contact_indexs = self.df.loc[self.df.label.diff() > 0.1, :].index
        end_contact_indexs = self.df.loc[self.df.label.diff() < -0.1, :].index - 1
        contact_indexs = [idx for idx, idx2 in zip(start_contact_indexs, end_contact_indexs) if idx2 - idx >= self.seq_num]

        sequences = []
        for contact_index in contact_indexs:
            end_point = contact_index + self.seq_num
            for step in range(contact_index, end_point, self.gap):
                window = self.df[self.selected_features][step - self.seq_num + 1:step + 1]
                sequences.append((window.values, self.df.label[step]))
        return sequences
    
    def _make_sequences(self):
        """Generate sequences over time"""
        sequences = []
        for step in range(self.seq_num,self.data.shape[0], self.gap):
            window = self.df[self.selected_features][step - self.seq_num:step]
            sequences.append((window.values, self.df.label[step-1]))
        
        return sequences

    def _get_specific_class(self, desired_label):
        filtered_data = [(seq, label) for seq, label in self.sequences if label == desired_label]
        return filtered_data


    def __len__(self):
        """Return the total number of sequences in the dataset."""
        return len(self.sequences)

    def __getitem__(self, idx):
        """
        Retrieve a single sequence and label.
        
        Args:
            idx (int): Index of the sequence to retrieve.
            
        Returns:
            (tuple): (features, target) where target is the label for classification.
        """
        #TODO: multiple features should be reshaped.
        features, target = self.sequences[idx]
        features = features.T if isinstance(features, torch.Tensor) else torch.tensor(features, dtype=torch.float32).T.clone().detach()
        target = target if isinstance(target, torch.Tensor) else torch.tensor(target, dtype=torch.long).clone().detach()
        return features, target


class LoadDatasets(Dataset):
    def __init__(self, data_path:str, dict_label = None):
        """
        Load sequential dataset from a directory structure with labeled subdirectories.

            Expected directory structure:
            
            path/to/data/
                ├── class_0/
                │   ├── sample1.pkl 
                │   ├── sample2.csv
                │   └── ...
                ├── class_1/
                │   ├── sample1.csv
                │   ├── sample2.pkl
                │   └── ...
                └── ...

            Label mapping example:
            
            dict_label = {'class_0': 0, 'class_1': 1, ... }
        
        Args:
            data_path (str): Path to the data directory.
            dict_label (dict, optional): Dictionary mapping class folder names to labels.
        """
        if dict_label is None:
            dict_label = {'a': 7, 'b': 6, 'c': 5, 'd': 4, 'e': 3, 'f': 2, 'g': 1}
            #dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1}

        
        self.samples = []
        self.class_to_idx = {}

        # Scan data_path for subdirectories        
        for class_name in sorted(os.listdir(data_path)):
            class_dir = os.path.join(data_path, class_name)
            if os.path.isdir(class_dir) and class_name in dict_label:
                label = dict_label[class_name]  # Look up label
                self.class_to_idx[class_name] = label
                for file_name in os.listdir(class_dir):
                    file_path = os.path.join(class_dir, file_name)
                    if os.path.isfile(file_path):
                        self.samples.append((file_path, label))
                        
    def __len__(self):
        """Return the total number of samples."""
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Retrieve a single sample at the specified index.
        
        Args:
            idx (int): Index of the sample to retrieve.
            
        Returns:
            tuple: (file_path, label)
        """
        seq_path, label = self.samples[idx]
        return seq_path, label

# 2. load AI models

In [2]:
import argparse
import os
import time
import torch
import numpy as np
import random
import torch.optim as optim
import torch.nn as nn
from torchmetrics import ConfusionMatrix, Accuracy
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

   

class cnnLSTM(nn.Module):
    def __init__(self, num_features_joints=28, hidden_size=32, num_layers=3, dropout=0.5, bidirectional=False):
        super(cnnLSTM, self).__init__()
        self.normalization = nn.LayerNorm(num_features_joints)#ZScoreNormalization()
        
         # Define the 1D CNN layers
        self.cnn1 = nn.Conv1d(in_channels=num_features_joints, out_channels=256, kernel_size=1, padding='same',padding_mode='circular')
        self.cnn2 = nn.Conv1d(in_channels=256, out_channels=512, kernel_size=2, padding='same',padding_mode='circular')
        self.cnn3 = nn.Conv1d(in_channels=512, out_channels=256, kernel_size=4, padding='same',padding_mode='circular')
        
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(512)
        self.bn3 = nn.BatchNorm1d(256)
        
        self.relu = nn.LeakyReLU() #nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout_cnn = nn.Dropout(0.3)
        
        
        # Define the LSTM layer
        self.lstm = nn.LSTM(
            input_size=256//2,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0  # Dropout only if num_layers > 1
        )

        lstm_output_size = hidden_size * 2 if bidirectional else hidden_size
        self.normalization_lstm = nn.LayerNorm(lstm_output_size)
        
        # Fully connected layer
        self.fc = nn.Linear(lstm_output_size, 1)

        # Dropout for regularization
        self.dropout_fc = nn.Dropout(dropout)
        
        

    def forward(self, input, freeze_last_layer=False):
        # Normalize input (batch_size, sequence_length, input_size)
        self.lstm.flatten_parameters()

        normalized_input = self.normalization(input)

        # Reshape for CNN: (batch_size, in_channels, sequence_length)
        cnn_input = normalized_input.permute(0, 2, 1)
        
        # Apply CNN layers
        cnn_out = self.relu(self.bn1(self.cnn1(cnn_input)))#, seq_pose = 1))
        cnn_out = self.relu(self.bn2(self.cnn2(cnn_out)))#, seq_pose = 1))
        cnn_out = self.relu(self.bn3(self.cnn3(cnn_out)))#, seq_pose = 1))
        cnn_out = cnn_out.permute(0,2,1)
        cnn_out = self.pool(cnn_out)

        # Reshape for LSTM: (batch_size, sequence_l98pength, input_size)

        lstm_out, _ = self.lstm(cnn_out)
        self.normalization_lstm(lstm_out)

        # Apply dropout
        lstm_out = self.dropout_fc(lstm_out)
        
        # Fully connected layer

        joint_step_outputs = self.fc(lstm_out).squeeze(-1)  # (batch_size, joint_dof, 1) -> (batch_size, joint_dof)
        
        return joint_step_outputs
    
    def prediction(self, input):
        device = input.device
        output = self.forward(input)
        
        # Check if all elements are masked (-inf) for each sample
        nocontact_masked = (output <= 0).all(dim=1)  # True if all entries are masked

        # Find the index of the max value (ignoring masked ones)
        predictions = torch.argmax(output, dim=1)+1  # Get index of max valid value
        predictions[nocontact_masked] = 0
        return predictions


# 3. Training

In [4]:
# functions
import copy
import plotly.express as px
import chart_studio.plotly as py
import plotly.graph_objs as go
from plotly.offline import iplot, init_notebook_mode
# Using plotly + cufflinks in offline mode
import cufflinks
cufflinks.go_offline(connected=True)
init_notebook_mode(connected=True)
from collections import Counter
def create_hierarchical_labels(contact_indices, num_links):
    """
    Generate hierarchical labels for a batch of contact indices.
    
    Args:
        contact_indices (Tensor): A tensor of shape [batch_size] with the contact index for each example in the batch.
        num_links (int): The number of links (length of the output vector).
    
    Returns:
        Tensor: A tensor of shape [batch_size, num_links] containing hierarchical labels for each example.
    """
    batch_size = contact_indices.size(0)  # Get batch size
    y_true = torch.zeros(batch_size, num_links, dtype=torch.float32, device=contact_indices.device)  # Initialize a tensor of zeros
    
    for i in range(batch_size):
        contact_index = contact_indices[i]
        if contact_index>0:
            y_true[i, contact_index-1 ] = 1  # Set the values up to contact_index to 1
            #y_true[i, 0:contact_index-1 ] = 0.8  # Set the values up to contact_index to 1
    
    return y_true
def majority_voting_last_n(model_out, n):
    """
    Apply majority voting for each step considering the last `n` items.

    Args:
        model_out (pd.Series): Predicted classes for each step.
        n (int): Number of previous items (including the current one) to consider for voting.

    Returns:
        pd.Series: Smoothed predictions based on majority voting.
    """
    smoothed_predictions = model_out.copy()  # Copy to retain index
    for i in range(n, len(model_out), 1):
        # Define the window
        start_idx = i - n 
        end_idx = i  # Include the current item
        window = model_out.iloc[start_idx:end_idx]
        
        # Perform majority voting
        most_common = Counter(window).most_common(1)[0][0]
        smoothed_predictions.iloc[i-1] = most_common
    
    return smoothed_predictions

def train_loop(model, dataset_loader,selected_features, lr, n_epochs, freeze_last_layer=False): #--> 
    model.train()
    # Use Adam optimizer and CrossEntropyLoss as the loss function
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    # Initialize learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, verbose=True)
    loss_seq = []
    lr_seq = []
    # Training loop
    for epoch in range(n_epochs):
        running_loss = []
        for trial_dataset_path, label in dataset_loader:
            
            data = LoadSeqDataset(file_path = trial_dataset_path[0], label= label[0],  selected_features= selected_features, seq_num=seq_num, gap=gap)
            
            dataloader = DataLoader(data, batch_size=batch_size, shuffle=True)
        
            for X_batch, y_batch in dataloader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device) # Move data to device
                optimizer.zero_grad()
                # Generate noise and shift as PyTorch tensors
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                
                loss.backward()
                optimizer.step()
                running_loss.append(loss.cpu().detach().numpy())

        avg_loss = np.mean(running_loss)
        print(f"Epoch: {epoch + 1}/{n_epochs} - learning rate: {optimizer.param_groups[0]['lr']:.5f}, classification loss: {avg_loss:.4f}")
        # Update the scheduler with the average loss
        scheduler.step(avg_loss)
        current_lr = optimizer.param_groups[0]['lr']
        if current_lr < lr_threshold:
            print(f"Learning rate has dropped below the threshold of {lr_threshold}. Stopping training.")
            break
        loss_seq.append(avg_loss)
        lr_seq.append(current_lr)

    return model, loss_seq, lr_seq

def plot_loss(loss_seq, lr_seq):
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot loss on the primary y-axis
    ax1.plot(loss_seq, label='Loss', color='blue', marker='o')
    ax1.set_xlabel('Epochs', fontsize=14)
    ax1.set_ylabel('Loss', fontsize=14)
    ax1.tick_params(axis='y', labelcolor='blue')
    
    # Set y-axis limits for loss to include 0
    ax1.set_ylim(bottom=0, top=max(loss_seq) * 1.05)  # Adjust top limit as needed

    ax1.grid(True)

    # Create a secondary y-axis
    ax2 = ax1.twinx()
    ax2.plot(lr_seq, label='Learning Rate', color='red', marker='o')
    ax2.set_ylabel('Learning Rate', fontsize=14)
    ax2.tick_params(axis='y', labelcolor='red')

    # Set y-axis limits for learning rate to include 0
    ax2.set_ylim(bottom=0, top=max(lr_seq) * 1.05)  # Adjust top limit as needed

    # Add legends
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc='best')

    # Improve layout
    plt.tight_layout()

    # Show the plot
    plt.show()


def validation(dataloaders, names, model,print_data = False):
    model.eval()
    accuracies = []
    with torch.no_grad():
        confusionMatrix = ConfusionMatrix(task="multiclass", num_classes=num_classes)
        accuracy_metric = Accuracy()
        for i in range(len(dataloaders)):
            y_pred, y_test = get_output(dataloaders[i], model, device)
            accuracy_metric.update(y_pred, y_test)
            accuracy = accuracy_metric.compute()
            if print_data: 
                print("Accuracy on ", names[i], ': ', accuracy)
            accuracies.append(accuracy.item()*100)  # Collect accuracy for plotting
            accuracy_metric.reset()
    return accuracies

# Bar chart plotting function
def plot_accuracies(pre_accuracies, post_accuracies, dataloader_names, model_name):
    labels = dataloader_names
    bar_width = 0.35
    index = range(len(labels))

    # Plotting the barchart
    plt.figure(figsize=(10, 6))
    plt.grid(True)

    bars1 = plt.bar(index, pre_accuracies, bar_width, label='Pre-DomainAdaptation Accuracy', color='blue')
    bars2 = plt.bar([i + bar_width for i in index], post_accuracies, bar_width, label='Post-DomainAdaptation Accuracy', color='green')

    # Set y-axis limits from 0 to 100
    plt.ylim(0, 120)

    # Adding labels, title, and legend
    plt.xlabel('Test Dataset', fontsize=14)
    plt.ylabel('Accuracy (%)', fontsize=14)  # Indicating accuracy as a percentage
    plt.title(f'Accuracy Comparison for {model_name}', fontsize=16)
    plt.xticks([i + bar_width / 2 for i in index], labels)
    plt.legend()

    # Adding accuracy values on top of the bars
    for bar in bars1:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, yval + 1, f'{yval:.2f}', ha='center', fontsize=12)

    for bar in bars2:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, yval + 1, f'{yval:.2f}', ha='center', fontsize=12)

    # Show the plot
    plt.tight_layout()
    plt.show()


In [6]:
# hyperparameters
import pickle, logging
accuracy_metric = Accuracy()

loss_fn = nn.CrossEntropyLoss()
loss_fn = nn.BCEWithLogitsLoss()
#loss_fn = nn.MultiLabelSoftMarginLoss()

lrs = [0.04, 0.002]
lr_threshold = 0.0005
n_epochs = [35, 20]
n = 14  # Window size for majority voting
batch_size = 71
data_name, dof = 'franka_main', 7 #franka_main, ur5
split_rate =0.75
torch.manual_seed(2020)
np.random.seed(2020)
random.seed(2020)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')
dataset_info = {os.getcwd().replace('pipelines', '') + f'/dataset/{data_name}/labeled_data/': dof}
logging.basicConfig(
    level=logging.INFO,  # Set the logging level to INFO
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),  # Print log to console
        logging.FileHandler(os.getcwd() + f'/trained_models/{data_name}/contact_localization/{data_name}_training_log_batch_size{batch_size}_{time.time()}.txt')#num_layers{num_layers}_hidden_size{hidden_size}.txt')  # Save log to a single file
    ]
)
dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]


for num_layers in [1,2,3]:
    for hidden_size in [32, 64, 128, 256]:
        for seq_num in [30, 50, 80, 100, 150, 200]:
            for gap in [3, 5, 10, 15]:
                # Load the dataset from the file _half_0_75_0_3
                with open(f'{main_path}/dataset/{data_name}/pickleDatasets/{data_name}_feature_e_gap_{gap}_splitRate_{split_rate}_seqNum_{seq_num}.pickle', 'rb') as f:
                    master_dataset = pickle.load(f)

                # Training
                train_dataloader = DataLoader(master_dataset, batch_size=batch_size, shuffle=True)

                # Build the model
                #model_lstmBlock = lstmBlock(num_features_joints=seq_num, num_layers=num_layers, hidden_size=64, dropout=0.2, bidirectional=True)
                model_cnnLSTM = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
                models = [model_cnnLSTM, model_cnnLSTM]
                models_names = ['model_lstmBlock',  'model_cnnLSTM',]
                for model in models:
                    model.to(device)
                    
                for ii in [1]:#range(len(models)):
                    model = models[ii]
                    #model.lstm.flatten_parameters()
                    
                    lr = lrs[ii]
                    logging.info(f'------------  {models_names[ii]} , num_layers = {num_layers}, hidden_size={hidden_size} seq_num = {seq_num}, gap = {gap}, --------------')
                    #  training on source robot
                    #model, loss_seq, lr_seq = train_loop(model, train_datasetloader,selected_features, lr, n_epochs[ii])
                    model.train()
                    # Use Adam optimizer and CrossEntropyLoss as the loss function
                    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
                    # Initialize learning rate scheduler
                    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, verbose=True)
                    loss_seq = []
                    lr_seq = []
                    # Training loop
                    for epoch in range(n_epochs[ii]):
                        running_loss = []
                        
                        for X_batch, y_batch in train_dataloader:
                            X_batch, y_batch = X_batch.to(device), y_batch.to(device) # Move data to device
                            mask = y_batch != 0
                            X_batch = X_batch[mask]
                            y_batch = y_batch[mask]
                            if y_batch.shape[0]>0:
                                optimizer.zero_grad()
                                y_pred = model(X_batch)
                                y_batch = create_hierarchical_labels(y_batch, dof)
                                loss = loss_fn(y_pred, y_batch)            
                                loss.backward()
                                optimizer.step()
                                running_loss.append(loss.cpu().detach().numpy())

                        avg_loss = np.mean(running_loss)
                        #print(f"Epoch: {epoch + 1}/{n_epochs} - learning rate: {optimizer.param_groups[0]['lr']:.5f}, classification loss: {avg_loss:.4f}")
                        # Update the scheduler with the average loss
                        scheduler.step(avg_loss)
                        current_lr = optimizer.param_groups[0]['lr']
                        if current_lr < lr_threshold:
                            print(f"Learning rate has dropped below the threshold of {lr_threshold}. Stopping training.")
                            break
                        loss_seq.append(avg_loss)
                        lr_seq.append(current_lr)
                        if avg_loss < 0.08:
                            print('early stopping <0.08!')
                            break
                    #logging.info(f'loss = {loss_seq}')
                    #logging.info(f'lr = {lr_seq}')
                    model.eval()
                    os.makedirs(f'{main_path}/pipelines/trained_models/{data_name}/contact_localization/{batch_size}/', exist_ok=True)
                    accuracies = []
                    # Loop through each dataset path and corresponding dof
                    for data_path, dof in dataset_info.items():
                        # Load the dataset for testing
                        testing_datasets = LoadDatasets(data_path, dict_label)
                        test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)
                        
                        for trial_dataset_path, label in test_datasetloader:
                            if 'link1' in trial_dataset_path[0]:
                                # Load the dataset for the specific trial
                                data = LoadSeqDataset(file_path=trial_dataset_path[0], label=label[0], 
                                                    selected_features=selected_features, seq_num=seq_num, gap=1)
                                data.sequences = data.sequences[len(data)//2:-1]
                                # Initialize an empty dataframe if not done before
                                df = pd.DataFrame(columns=["time", "label", "model_out", "probability", "majority_voting"])

                                # Create a DataLoader for this specific trial data
                                test_loader = DataLoader(data, batch_size=len(data), shuffle=False)

                                # Iterate through the DataLoader to make predictions
                                for batch_idx, (seqs, labels) in enumerate(test_loader):
                                    seqs = seqs.float().to(device)  # Convert sequences to float and move to device

                                    # Perform the prediction without gradient computation
                                    with torch.no_grad():
                                        predictions = model(seqs)
                                        predictions = torch.argmax(predictions, dim=1) + 1

                                    # Fill the dataframe with the results
                                    df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                                    df['label'] = labels.cpu().numpy()  # Convert label to numpy
                                    df['model_out'] = predictions.cpu().detach().numpy()  # Convert predictions to numpy
                                    mask = df.label == 0
                                    df.model_out[mask]= 0
                                    df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                                # Plotting using Plotly (assuming you have plotly installed)
                                df.iplot(x='time', y=['model_out', 'majority_voting', 'label'], colors=[ 'lightblue', 'darkblue', 'red'], title='Link ' + str(labels.max().item()))
                                accuracy_metric.update(torch.tensor(df['majority_voting'].values), torch.tensor(df['label'].values))
                                accuracy = accuracy_metric.compute()
                                accuracies.append(accuracy.item()*100)
                                accuracy_metric.reset()

                                '''confmat = ConfusionMatrix(task="multiclass", num_classes=dof+1)
                                cm = confmat(torch.tensor(df['majority_voting'].values), torch.tensor(df['label'].values))
                                logging.info(f'\n{cm}')'''
                                break
                        logging.info(f'Accuracy on the test data = {accuracies}')
                        torch.save(model.state_dict(), f'{main_path}/pipelines/trained_models/{data_name}/contact_localization/{batch_size}/numLayer{num_layers}_hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}_accuracy{accuracies[0]:.2f}')
                        break 
                    break
                break
            break
        break
    break

Using GPU: Quadro RTX 8000


2025-09-09 11:20:21,411 - INFO - ------------  model_cnnLSTM , num_layers = 1, hidden_size=32 seq_num = 30, gap = 3, --------------


2025-09-09 11:25:27,346 - INFO - Accuracy on the test data = [61.38306260108948]


## accuray calucation

In [8]:
from sklearn.metrics import classification_report, confusion_matrix
import logging
data_name, dof, batch_size = 'franka_main', 7 , 71


best_models_index, folder_name , data_name, dof  = ([   [1, 64, 3, 50],
                                                        [1, 64, 3, 80],
                                                        [1, 64, 3, 100],
                                                        [1, 64, 5, 50],
                                                        [1, 64, 5, 80],
                                                        [1, 64, 5, 100],
                                                        [1, 128, 3, 50],
                                                        [1, 128, 3, 80],
                                                        [1, 128, 3, 100],
                                                        [1, 128, 5, 50],
                                                        [1, 128, 5, 80],
                                                        [1, 128, 5, 100] ], f'pipelines/trained_models/{data_name}/contact_localization/{batch_size}/', f'{data_name}', dof)

logging.info(f'trained model on {folder_name}, tested on {data_name}, with {dof} links')
'''
best_models_index = []
for num_layers in [1, 2, 3]:
    for hidden_size in [32, 64, 128, 256]:
        for seq_num in [30, 50, 80, 100, 150, 200]:
            for gap in [3, 5, 10, 15]:
                best_models_index.append([num_layers, hidden_size, gap, seq_num])
'''

n = 14  # Window size for majority voting

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')


dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]

# Load dataset ONCE outside the model loop
testing_datasets = LoadDatasets(os.getcwd().replace('pipelines', '') + f'/dataset/{data_name}/labeled_data/', dict_label)
test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)

file_predictions = {}
for link in dict_label.keys():
    for trial_dataset_path, label in test_datasetloader:
        if link in trial_dataset_path[0]:
            print(link)
            for num_layers, hidden_size, gap, seq_num in best_models_index:
                for model_name in os.listdir(f'{main_path}/{folder_name}/'):                   
                    if f'numLayer{num_layers}_hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}' in model_name:
                        if model_name not in file_predictions:
                            file_predictions[model_name] = {'y_pred': [], 'y_true': []}
                        # Load and evaluate model
                        model = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
                        model.to(device)
                        model.load_state_dict(torch.load(f'{main_path}/{folder_name}/{model_name}'))
                        model.eval()
                        data = LoadSeqDataset(
                            file_path=trial_dataset_path[0], label=label[0], 
                            selected_features=selected_features, seq_num=seq_num, gap=1
                        )
                        #data.sequences = data.sequences[len(data)//2:-1]
                        data.sequences = data.sequences[(len(data)-len(data)//2):-1]

                        test_loader = DataLoader(data, batch_size=len(data), shuffle=False)
                        # Run inference
                        df= pd.DataFrame()
                        for batch_idx, (seqs, labels) in enumerate(test_loader):
                            seqs = seqs.float().to(device)
                            with torch.no_grad():
                                predictions = model(seqs)
                                predictions = torch.argmax(predictions, dim=1) + 1

                            df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                            df['label'] = labels.cpu().numpy()
                            df['model_out'] = predictions.cpu().detach().numpy()
                            df.model_out[df.label == 0] = 0
                            df['majority_voting'] = majority_voting_last_n(df['model_out'], n)
                            
                        #df.iplot(x='time', y=['model_out', 'majority_voting', 'label'], colors=[ 'lightblue', 'darkblue', 'red'], title='Link ' + str(labels.max().item()))
                        # Collect predictions for current model
                        y_true_filtered = df.label[(df.label != 0) & (df.majority_voting != 0)].tolist()
                        y_pred_filtered = df.majority_voting[(df.label != 0) & (df.majority_voting != 0)].tolist()
                        file_predictions[model_name]['y_pred'].extend(y_pred_filtered)
                        file_predictions[model_name]['y_true'].extend(y_true_filtered)
                        break  # Process only first trial per link
                    
            break

for model_name in file_predictions.keys():
    y_pred = file_predictions[model_name]['y_pred']
    y_true = file_predictions[model_name]['y_true']
    matrix = confusion_matrix(y_pred=y_pred, y_true=y_true)

    total_samples = sum(sum(row) for row in matrix)
    logging.info(f'{model_name}, total samples: {total_samples}')
    #logging.info(f'\n {matrix}')
    true_positives = sum(matrix[i][i] for i in range(len(matrix)))

    overall_accuracy = (true_positives / total_samples) * 100
    failure_rate = [(sum(matrix[i]) - matrix[i][i]) / sum(matrix[i]) * 100 for i in range(len(matrix))]
    logging.info(f'acuracy= {overall_accuracy}, detection failure (links): {failure_rate}')


2025-09-09 11:30:10,159 - INFO - trained model on pipelines/trained_models/franka_main/contact_localization/71/, tested on franka_main, with 7 links


Using GPU: Quadro RTX 8000
link7
link6
link5
link4
link3
link2
link1


2025-09-09 11:30:19,739 - INFO - numLayer1_hiddenSize32_seq_num30_gap3_accuracy61.38, total samples: 3949
2025-09-09 11:30:19,740 - INFO - acuracy= 78.65282349962015, detection failure (links): [64.69594594594594, 3.322784810126582, 28.176795580110497, 13.704496788008566, 23.765432098765434, 2.7732463295269167, 11.233480176211454]


## plots

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

data_name, dof = 'franka_main', 7 #franka_main, ur5

best_models_index, batch_size, folder_name , data_name, dof  = ([   [1, 32, 3, 30],
                                                                    [1, 64, 3, 80],
                                                                    [1, 64, 3, 100],
                                                                    [1, 64, 5, 50],
                                                                    [1, 64, 5, 80],
                                                                    [1, 64, 5, 100],
                                                                    [1, 128, 3, 50],
                                                                    [1, 128, 3, 80],
                                                                    [1, 128, 3, 100],
                                                                    [1, 128, 5, 50],
                                                                    [1, 128, 5, 80],
                                                                    [1, 128, 5, 100] ], 64, f'pipelines/trained_models/{data_name}/contact_localization/{batch_size}/', data_name, dof)


# hyperparameters

n = 14  # Window size for majority voting

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')

dataset_info = {main_path + f'/dataset/{data_name}/labeled_data/': dof}

dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]


counter = 1
links = ['link7', 'link6', 'link5', 'link4', 'link3', 'link2', 'link1']
for num_layers, hidden_size, gap, seq_num in best_models_index:
    for model_name in os.listdir(f'{main_path}/{folder_name}/'):
        if f'hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}' in model_name:
            print(counter,')  ' , num_layers, hidden_size, gap, seq_num)
            counter +=1
            model = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
            model.to(device)
            model.load_state_dict(torch.load(f'{main_path}/{folder_name}/{model_name}'))
            model.eval()
            y_pred, y_true= [], []

            for data_path, dof in dataset_info.items():
                # Load the dataset for testing
                testing_datasets = LoadDatasets(data_path, dict_label)
                test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)
                y_true, y_pred = [], []
                for link in links:
                    # Iterate through the dataset (trials)
                    for trial_dataset_path, label in test_datasetloader:
                        if link in trial_dataset_path[0]:
                            # Load the dataset for the specific trial
                            data = LoadSeqDataset(file_path=trial_dataset_path[0], label=label[0], 
                                                selected_features=selected_features[0:dof], seq_num=seq_num, gap=1)
                            data.sequences = data.sequences[len(data)//2:-1]
                            # Initialize an empty dataframe if not done before
                            df = pd.DataFrame(columns=["time", "label", "model_out", "probability", "majority_voting"])

                            # Create a DataLoader for this specific trial data
                            test_loader = DataLoader(data, batch_size=len(data), shuffle=False)

                            # Iterate through the DataLoader to make predictions
                            for batch_idx, (seqs, labels) in enumerate(test_loader):
                                seqs = seqs.float().to(device)  # Convert sequences to float and move to device

                                # Perform the prediction without gradient computation
                                with torch.no_grad():
                                    predictions = model(seqs)
                                    predictions = torch.argmax(predictions, dim=1)+1

                                # Fill the dataframe with the results
                                df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                                df['label'] = labels.cpu().numpy()  # Convert label to numpy
                                df['model_out'] = predictions.cpu().detach().numpy()  # Convert predictions to numpy
                                df.model_out[df.label==0] = 0
                                df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                            # Plotting using Plotly (assuming you have plotly installed)
                            df.iplot(x='time', y=['model_out', 'majority_voting', 'label'], 
                                    colors=[ 'lightblue', 'darkblue', 'red'], 
                                    title=f'{link}_{model_name}')
                            y_true = y_true + df.label[(df.label != 0) & (df.majority_voting != 0)].tolist()
                            y_pred = y_pred + df.majority_voting[(df.label != 0) & (df.majority_voting != 0)].tolist()

                            

                            break
                # Pass to sklearn safely
                cm = confusion_matrix(y_pred=y_pred, y_true=y_true)
                print(cm)
                #logging.info(f'Testing {hidden_size},{gap}, {seq_num}, {model_name}, total classified samples:{sum(sum(cm))}')
                #logging.info(f'\n{cm}')

Using GPU: Quadro RTX 8000
1 )   1 32 3 30


[[209  13   0   0  41 328   0]
 [  0 611   3   0   2  16   0]
 [  0  86 390  50  18   0   0]
 [  0   0  59 403   3   2   0]
 [  0   0   0   6 494 138  11]
 [  0   0   0   0  17 597   0]
 [  0   0   0   0  15  36 403]]


# 4. Transfer Learning

## Fine Tuning

In [ ]:
accuracy_metric = Accuracy()
loss_fn = nn.MultiLabelSoftMarginLoss()
loss_fn = nn.BCEWithLogitsLoss()
#loss_fn = nn.CrossEntropyLoss()
lrs = [0.04, 0.002]
lr_threshold = 0.0001
n_epochs = [35, 20]
n = 14  # Window size for majority voting
batch_size=71
best_models_index, folder_name, data_name, batch_size, dof = ([ [1, 64, 3, 50],
                                                                [1, 64, 3, 80],
                                                                [1, 64, 3, 100],
                                                                [1, 64, 5, 50],
                                                                [1, 64, 5, 80],
                                                                [1, 64, 5, 100],
                                                                [1, 128, 3, 50],
                                                                [1, 128, 3, 80],
                                                                [1, 128, 3, 100],
                                                                [1, 128, 5, 50],
                                                                [1, 128, 5, 80],
                                                                [1, 128, 5, 100] ], f'pipelines/trained_models/franka_main/contact_localization/{batch_size}/', 'ur5', batch_size, 6)
logging.info(f'fine-tuning {folder_name} on {data_name} with {dof} links')
torch.manual_seed(2020)
np.random.seed(2020)
random.seed(2020)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')
dataset_info = {main_path + f'/dataset/{data_name}/labeled_data/': dof}
logging.basicConfig(
    level=logging.INFO,  # Set the logging level to INFO
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),  # Print log to console
        logging.FileHandler(os.getcwd() + f'/trained_models/{data_name}/contact_localization/Fine_tuning_{data_name}_training_log_batch_size{batch_size}.txt')#num_layers{num_layers}_hidden_size{hidden_size}.txt')  # Save log to a single file
    ]
)
dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]

for num_layers, hidden_size, gap, seq_num in best_models_index:
        # Load the dataset from the file _half__1_0_3
        with open(f'{main_path}/dataset/{data_name}/pickleDatasets/{data_name}_feature_e_gap_{gap}_splitRate_{split_rate}_seqNum_{seq_num}.pickle', 'rb') as f:
            master_dataset = pickle.load(f)
    
        # Training
        train_dataloader = DataLoader(master_dataset, batch_size=batch_size, shuffle=True)

        for model_name in os.listdir(f'{main_path}/{folder_name}/'):
            if f'numLayer{num_layers}_hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}' in model_name:
                # Build the model
                print(model_name)
                model = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
                model.to(device)
                model.load_state_dict(torch.load(f'{main_path}/{folder_name}/{model_name}'))
                model.train()
                    
                lr = lrs[1]
                logging.info(f'------------  {models_names[ii]} , num_layers = {num_layers}, hidden_size={hidden_size} seq_num = {seq_num}, gap = {gap}, --------------')
                optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, verbose=True)
                loss_seq = []
                lr_seq = []
                # Training loop
                for epoch in range(n_epochs[1]):
                    running_loss = []
                    
                    for X_batch, y_batch in train_dataloader:
                        X_batch, y_batch = X_batch.to(device), y_batch.to(device) # Move data to device
                        mask = y_batch !=0
                        y_batch = y_batch[mask]
                        X_batch = X_batch[mask]
                        if y_batch.shape[0] >0 :
                            optimizer.zero_grad()
                            y_pred = model(X_batch)
                            y_batch = create_hierarchical_labels(y_batch, dof)
                            loss = loss_fn(y_pred, y_batch)            
                            loss.backward()
                            optimizer.step()
                            running_loss.append(loss.cpu().detach().numpy())

                    avg_loss = np.mean(running_loss)
                    print(f"Epoch: {epoch + 1}/{n_epochs} - learning rate: {optimizer.param_groups[0]['lr']:.5f}, classification loss: {avg_loss:.4f}")
                    # Update the scheduler with the average loss
                    scheduler.step(avg_loss)
                    current_lr = optimizer.param_groups[0]['lr']
                    if current_lr < lr_threshold:
                        print(f"Learning rate has dropped below the threshold of {lr_threshold}. Stopping training.")
                        break
                    loss_seq.append(avg_loss)
                    lr_seq.append(current_lr)
                    if avg_loss < 0.08:
                        print('early stopping <0.08!')
                        break
                #logging.info(f'loss = {loss_seq}')
                #logging.info(f'lr = {lr_seq}')
                model.eval()
                os.makedirs(f'{main_path}/pipelines/trained_models/{data_name}/contact_localization/fine_tuned{batch_size}/', exist_ok=True)
                accuracies = []
                # Loop through each dataset path and corresponding dof
                for data_path, dof in dataset_info.items():
                    # Load the dataset for testing
                    testing_datasets = LoadDatasets(data_path, dict_label)
                    test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)
                    
                    for trial_dataset_path, label in test_datasetloader:
                        if 'link1' in trial_dataset_path[0]:
                            # Load the dataset for the specific trial
                            data = LoadSeqDataset(file_path=trial_dataset_path[0], label=label[0], 
                                                selected_features=selected_features[0:dof], seq_num=seq_num, gap=1)
                            data.sequences = data.sequences[(len(data)-len(data)//2):len(data)]
                            # Initialize an empty dataframe if not done before
                            df = pd.DataFrame(columns=["time", "label", "model_out", "probability", "majority_voting"])

                            # Create a DataLoader for this specific trial data
                            test_loader = DataLoader(data, batch_size=len(data), shuffle=False)

                            # Iterate through the DataLoader to make predictions
                            for batch_idx, (seqs, labels) in enumerate(test_loader):
                                seqs = seqs.float().to(device)  # Convert sequences to float and move to device

                                # Perform the prediction without gradient computation
                                with torch.no_grad():
                                    predictions = model(seqs)
                                    predictions = torch.argmax(predictions, dim=1)+1

                                # Fill the dataframe with the results
                                df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                                df['label'] = labels.cpu().numpy()  # Convert label to numpy
                                df['model_out'] = predictions.cpu().detach().numpy()  # Convert predictions to numpy
                                df.model_out[df.label==0] = 0
                                df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                            # Plotting using Plotly (assuming you have plotly installed)
                            df.iplot(x='time', y=['model_out', 'majority_voting', 'label'], colors=[ 'lightblue', 'darkblue', 'red'], title='Link ' + str(labels.max().item()))
                            accuracy_metric.update(torch.tensor(df['majority_voting'].values), torch.tensor(df['label'].values))
                            accuracy = accuracy_metric.compute()
                            accuracies.append(accuracy.item()*100)
                            accuracy_metric.reset()

                            '''confmat = ConfusionMatrix(task="multiclass", num_classes=dof+1)
                            cm = confmat(torch.tensor(df['majority_voting'].values), torch.tensor(df['label'].values))
                            logging.info(f'\n{cm}')'''
                            break
                    logging.info(f'Accuracy on the test data = {accuracies}')
                    torch.save(model.state_dict(), f'{main_path}/pipelines/trained_models/{data_name}/contact_localization/fine_tuned{batch_size}/numLayer{num_layers}_hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}_accuracy{accuracies[0]:.2f}.pth')
                    break 

2025-09-09 11:47:03,574 - INFO - fine-tuning pipelines/trained_models/franka_main/contact_localization/62/ on ur5 with 6 links
2025-09-09 11:47:03,628 - INFO - ------------  model_cnnLSTM , num_layers = 1, hidden_size=128 seq_num = 100, gap = 5, --------------


Using GPU: Quadro RTX 8000
numLayer1_hiddenSize128_seq_num100_gap5_accuracy91.93
Epoch: 1/[35, 20] - learning rate: 0.00200, classification loss: 0.4187
Epoch: 2/[35, 20] - learning rate: 0.00200, classification loss: 0.1947
Epoch: 3/[35, 20] - learning rate: 0.00200, classification loss: 0.1162
Epoch: 4/[35, 20] - learning rate: 0.00200, classification loss: 0.0759
early stopping <0.08!


2025-09-09 11:47:07,649 - INFO - Accuracy on the test data = [87.78396844863892]
2025-09-09 11:47:07,671 - INFO - ------------  model_cnnLSTM , num_layers = 1, hidden_size=128 seq_num = 100, gap = 5, --------------


numLayer1_hiddenSize128_seq_num100_gap5_accuracy91.61
Epoch: 1/[35, 20] - learning rate: 0.00200, classification loss: 0.3921
Epoch: 2/[35, 20] - learning rate: 0.00200, classification loss: 0.1801
Epoch: 3/[35, 20] - learning rate: 0.00200, classification loss: 0.1086
Epoch: 4/[35, 20] - learning rate: 0.00200, classification loss: 0.0698
early stopping <0.08!


2025-09-09 11:47:11,732 - INFO - Accuracy on the test data = [84.22631621360779]


## accuracy

In [20]:
from sklearn.metrics import classification_report, confusion_matrix
#data_name, dof = 'franka_mindlab', 7
best_models_index, folder_name , data_name, dof  = ([   [1, 64, 3, 50],
                                                        [1, 64, 3, 80],
                                                        [1, 64, 3, 100],
                                                        [1, 64, 5, 50],
                                                        [1, 64, 5, 80],
                                                        [1, 64, 5, 100],
                                                        [1, 128, 3, 50],
                                                        [1, 128, 3, 80],
                                                        [1, 128, 3, 100],
                                                        [1, 128, 5, 50],
                                                        [1, 128, 5, 80],
                                                        [1, 128, 5, 100] ], f'pipelines/trained_models/{data_name}/contact_localization/fine_tuned{batch_size}/', data_name, dof)
logging.info(f'Fine-tuned source model with {data_name}, tested on {data_name}, with {dof} links')

#best_models_index = [[1, 128, 3, 100]]
# hyperparameters

n = 14  # Window size for majority voting

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')


dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]

# Load dataset ONCE outside the model loop
testing_datasets = LoadDatasets(os.getcwd().replace('pipelines', '') + f'/dataset/{data_name}/labeled_data/', dict_label)
test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)

file_predictions = {}
for link in dict_label.keys():
    for trial_dataset_path, label in test_datasetloader:
        if link in trial_dataset_path[0]:
            print(link)
            for num_layers, hidden_size, gap, seq_num in best_models_index:
                for model_name in os.listdir(f'{main_path}/{folder_name}/'):                   
                    if f'hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}' in model_name:
                        if model_name not in file_predictions:
                            file_predictions[model_name] = {'y_pred': [], 'y_true': []}
                        # Load and evaluate model
                        model = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
                        model.to(device)
                        model.load_state_dict(torch.load(f'{main_path}/{folder_name}/{model_name}'))
                        model.eval()
                        data = LoadSeqDataset(
                            file_path=trial_dataset_path[0], label=label[0], 
                            selected_features=selected_features, seq_num=seq_num, gap=1
                        )
                        #data.sequences = data.sequences[len(data)//2:-1]
                        data.sequences = data.sequences[(len(data)-len(data)//2):-1]

                        test_loader = DataLoader(data, batch_size=len(data), shuffle=False)
                        # Run inference
                        df= pd.DataFrame()
                        for batch_idx, (seqs, labels) in enumerate(test_loader):
                            seqs = seqs.float().to(device)
                            with torch.no_grad():
                                predictions = model(seqs)
                                predictions = torch.argmax(predictions, dim=1) + 1

                            df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                            df['label'] = labels.cpu().numpy()
                            df['model_out'] = predictions.cpu().detach().numpy()
                            df.model_out[df.label == 0] = 0
                            df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                        # Collect predictions for current model
                        y_true_filtered = df.label[(df.label != 0) & (df.majority_voting != 0)].tolist()
                        y_pred_filtered = df.majority_voting[(df.label != 0) & (df.majority_voting != 0)].tolist()
                        file_predictions[model_name]['y_pred'].extend(y_pred_filtered)
                        file_predictions[model_name]['y_true'].extend(y_true_filtered)
                        break  # Process only first trial per link
            break

for model_name in file_predictions.keys():
    y_pred = file_predictions[model_name]['y_pred']
    y_true = file_predictions[model_name]['y_true']
    matrix = confusion_matrix(y_pred=y_pred, y_true=y_true)

    total_samples = sum(sum(row) for row in matrix)
    logging.info(f'{model_name}, total samples: {total_samples}')
    #logging.info(f'\n {matrix}')
    true_positives = sum(matrix[i][i] for i in range(len(matrix)))

    overall_accuracy = (true_positives / total_samples) * 100
    failure_rate = [(sum(matrix[i]) - matrix[i][i]) / sum(matrix[i]) * 100 for i in range(len(matrix))]
    logging.info(f'acuracy= {overall_accuracy}, detection failure (links): {failure_rate}')


2025-09-09 11:48:30,278 - INFO - Fine-tuned source model with ur5, tested on ur5, with 6 links


Using GPU: Quadro RTX 8000
link6
link5
link4
link3
link2
link1


2025-09-09 11:48:48,328 - INFO - numLayer1_hiddenSize128_seq_num100_gap5_accuracy87.78, total samples: 4435
2025-09-09 11:48:48,328 - INFO - acuracy= 61.44306651634723, detection failure (links): [30.26086956521739, 12.517580872011253, 24.110671936758894, 45.359281437125745, 36.20178041543027, 82.51572327044026]


### plot


In [21]:

best_models_index, folder_name , data_name, dof  = ([   [1, 64, 3, 50],
                                                        [1, 64, 3, 80],
                                                        [1, 64, 3, 100],
                                                        [1, 64, 5, 50],
                                                        [1, 64, 5, 80],
                                                        [1, 64, 5, 100],
                                                        [1, 128, 3, 50],
                                                        [1, 128, 3, 80],
                                                        [1, 128, 3, 100],
                                                        [1, 128, 5, 50],
                                                        [1, 128, 5, 80],
                                                        [1, 128, 5, 100] ], f'pipelines/trained_models/{data_name}/contact_localization/fine_tuned{batch_size}/', data_name, dof)

#best_models_index = [[2, 128, 3, 200]]
# hyperparameters

n = 14  # Window size for majority voting

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name())

main_path = os.getcwd().replace('pipelines', '')

dataset_info = {
os.getcwd().replace('pipelines', '') + f'/dataset/{data_name}/labeled_data/': 7
}

dict_label = {'link7': 7, 'link6':6, 'link5':5, 'link4':4, 'link3':3, 'link2':2, 'link1':1, 'no_contact': 0}
selected_features = [f'e{i}' for i in range(dof)]#+[ f'de{i}' for i in range(robot_dof)]


counter = 1
links = ['link7', 'link6', 'link5', 'link4', 'link3', 'link2', 'link1']
for num_layers, hidden_size, gap, seq_num in best_models_index:
    for model_name in os.listdir(f'{main_path}/{folder_name}/'):
        if f'numLayer{num_layers}_hiddenSize{hidden_size}_seq_num{seq_num}_gap{gap}' in model_name:
            print(counter,')  ' , num_layers, hidden_size, gap, seq_num)
            counter +=1
            model = cnnLSTM(num_features_joints=seq_num, num_layers=num_layers, hidden_size=hidden_size, dropout=0.7, bidirectional=True)
            model.to(device)
            model.load_state_dict(torch.load(f'{main_path}/{folder_name}/{model_name}'))
            model.eval()
            y_pred, y_true= [], []

            for data_path, dof in dataset_info.items():
                # Load the dataset for testing
                testing_datasets = LoadDatasets(data_path, dict_label)
                test_datasetloader = DataLoader(testing_datasets, batch_size=1, shuffle=False)
                for link in links:
                    # Iterate through the dataset (trials)
                    for trial_dataset_path, label in test_datasetloader:
                        if link in trial_dataset_path[0]:
                            # Load the dataset for the specific trial
                            data = LoadSeqDataset(file_path=trial_dataset_path[0], label=label[0], 
                                                selected_features=selected_features[0:dof], seq_num=seq_num, gap=1)
                            data.sequences = data.sequences[len(data)//2:-1]
                            # Initialize an empty dataframe if not done before
                            df = pd.DataFrame(columns=["time", "label", "model_out", "probability", "majority_voting"])

                            # Create a DataLoader for this specific trial data
                            test_loader = DataLoader(data, batch_size=len(data), shuffle=False)

                            # Iterate through the DataLoader to make predictions
                            for batch_idx, (seqs, labels) in enumerate(test_loader):
                                seqs = seqs.float().to(device)  # Convert sequences to float and move to device

                                # Perform the prediction without gradient computation
                                with torch.no_grad():
                                    predictions = model.prediction(seqs)

                                # Fill the dataframe with the results
                                df['time'] = data.data.time[len(data.data.time)-len(data)-1:-1]
                                df['label'] = labels.cpu().numpy()  # Convert label to numpy
                                df['model_out'] = predictions.cpu().detach().numpy()  # Convert predictions to numpy
                                df['majority_voting'] = majority_voting_last_n(df['model_out'], n)

                            # Plotting using Plotly (assuming you have plotly installed)
                            df.iplot(x='time', y=['model_out', 'majority_voting', 'label'], 
                                    colors=[ 'lightblue', 'darkblue', 'red'], 
                                    title=f'{link}_{model_name}')
                            break

Using GPU: Quadro RTX 8000
1 )   1 128 5 100


2 )   1 128 5 100
